# Análisis Exploratorio (EDA) y Medición de KPIs - Servicio Ciudadano 1800
Este notebook se conecta al Data Warehouse (`ServicioCiudadanoDW`) para extraer los datos limpios y responder a los KPIs de negocio establecidos.

In [ ]:
import os
import pyodbc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

# Configurar conexión usando variables de entorno (Docker)
server = os.getenv('DB_SERVER', 'localhost')
database = os.getenv('DB_NAME', 'ServicioCiudadanoDW')
username = os.getenv('DB_USER', 'SA')
password = os.getenv('DB_PASSWORD', 'J0AhMXX4H0QBrRr8R1U098y8aA1@')
driver = os.getenv('DB_DRIVER', '{ODBC Driver 18 for SQL Server}')

connection_string = f'DRIVER={driver};SERVER={server};DATABASE={database};UID={username};PWD={password};TrustServerCertificate=yes;'

print("Conectando a BD...")
conn = pyodbc.connect(connection_string)

# Extraer datos limpios de la vista
query = "SELECT * FROM vw_v1_limpio"
df = pd.read_sql(query, conn)
conn.close()

print(f"Total de registros limpios: {len(df)}")
df.head()


In [ ]:
# KPI 1 & 2: Tasa de Recontacto (TR7) y FCR Estimado
tr7 = df['recontacto_7_dias'].mean() * 100
fcr = 100 - tr7

print(f"--- KPI 1: Tasa de Recontacto a 7 días (TR7) ---")
print(f"Valor Actual: {tr7:.2f}% (Meta: <= 22%)")
print(f"--- KPI 2: FCR Estimado ---")
print(f"Valor Actual: {fcr:.2f}% (Meta: >= 78%)")


In [ ]:
# KPI 3: Tasa de Recontacto por Canal
plt.figure(figsize=(10,5))
canal_recontacto = df.groupby('canal')['recontacto_7_dias'].mean() * 100
canal_recontacto.sort_values(ascending=False).plot(kind='bar', color='salmon')
plt.title('KPI 3: Tasa de Recontacto por Canal')
plt.ylabel('% Recontacto')
plt.axhline(46, color='red', linestyle='--', label='Límite de riesgo (46%)')
plt.legend()
plt.show()
